# Phase 6: Experiments and the Critical Adoption Share A*

Goal: sweep mu (price-impact coefficient) and phi (residual AR coefficient) at a fixed forecast window, and for each parameter combination locate the critical adoption share A* at which the proposal's mechanism has bitten. Produce two heatmaps and a summary table. CLAUDE.md describes phase 6 as the compact headline result of the project.

Reference: section 4.2 of the proposal for A* and the threshold metrics, section 3.5 for stochastic diffusion, and phase 4 for the residual-vs-realised R^2 distinction.

**A note on the threshold.** The proposal defines A* as the smallest A at which the rolling out-of-sample R^2 crosses zero, or net profit crosses zero. In this cost-free baseline the residual R^2 falls but stays positive (typically ~0.025 even at full adoption) and adopter mean profit stays positive (it grows with adoption, by analytical inspection). So an absolute zero crossing does not fire. The cells below use **relative** thresholds instead: A*_R2 is the smallest A at which the rolling residual R^2 has fallen to half its baseline value, and A*_phi is the smallest A at which the rolling effective phi has grown to 1.5x its baseline. Once transaction costs land in phase 7 the absolute zero-crossing definition becomes operational and we swap back.

In [ ]:
# Common run parameters.
N = 200
T = 8000
sigma_news = 0.01
sigma_q = 1.0
forecast_window = 250
forecast_p = 1
risk_scale = 0.001
q_cap = 0.05
eval_window = 1000
adoption_start_t = forecast_window + eval_window
adoption_pi = 3e-4   # slow stochastic diffusion: A ramps from 0 to ~0.87 over the run

# Sweep grid (lists; converted to numpy in the run cell).
mu_grid = [0.025, 0.05, 0.075, 0.10, 0.15]
phi_grid = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

# Number of seeds per (mu, phi) cell. 10 keeps the total under ~15 min.
num_seeds_phase6 = 10
base_seed = 2000

# Relative thresholds for A*. The R^2 threshold is hit when the rolling
# residual R^2 has dropped to half its baseline; the phi threshold is
# hit when the rolling effective phi has grown by 50%.
r2_threshold_factor = 0.5
phi_threshold_factor = 1.5

# Window for the baseline (low-adoption) average against which the relative
# thresholds are taken. Sits just after the rolling-metric warm-up.
baseline_lo = adoption_start_t
baseline_hi = adoption_start_t + 1500

fig_dir = "../results/figures"
data_dir = "../results/data"

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from reflexive_market import simulate
from reflexive_market.metrics import rolling_oos_r2, rolling_phi

In [ ]:
def find_a_star(adoption_share, metric, threshold, direction):
    """Smallest adoption share at which ``metric`` crosses ``threshold``.

    direction == 'below': smallest A where metric < threshold (used for R^2 erosion).
    direction == 'above': smallest A where metric > threshold (used for phi growth).
    Returns NaN if no crossing happens in the observable range.
    """
    if direction == "below":
        crossed = np.isfinite(metric) & (metric < threshold)
    else:
        crossed = np.isfinite(metric) & (metric > threshold)
    if not np.any(crossed):
        return np.nan
    return float(adoption_share[crossed].min())


mu_arr = np.array(mu_grid)
phi_arr = np.array(phi_grid)
n_phi = phi_arr.size
n_mu = mu_arr.size
a_star_R2 = np.full((n_phi, n_mu), np.nan)
a_star_phi = np.full((n_phi, n_mu), np.nan)
baseline_R2_grid = np.full((n_phi, n_mu), np.nan)
baseline_phi_grid = np.full((n_phi, n_mu), np.nan)

for i, phi_v in enumerate(phi_arr):
    for j, mu_v in enumerate(mu_arr):
        a_star_R2_seeds = []
        a_star_phi_seeds = []
        baseline_R2_seeds = []
        baseline_phi_seeds = []
        for s in range(num_seeds_phase6):
            seed_s = base_seed + 1000 * (i * n_mu + j) + s
            rng_s = np.random.default_rng(seed_s)
            out_s = simulate.run(
                T=T, N=N, mu=float(mu_v), phi=float(phi_v),
                sigma_news=sigma_news, sigma_q=sigma_q,
                rng=rng_s,
                forecast_window=forecast_window, forecast_p=forecast_p,
                risk_scale=risk_scale, q_cap=q_cap,
                adoption_pi=adoption_pi, adoption_delta=0.0,
                adoption_start_t=adoption_start_t,
            )
            A = out_s["adoption_share"]
            residual = out_s["returns"] - mu_v * out_s["demand"]
            R2 = rolling_oos_r2(residual, out_s["forecasts"], eval_window)
            phi_eff = rolling_phi(out_s["returns"], eval_window)

            base_R2 = float(np.nanmean(R2[baseline_lo:baseline_hi]))
            base_phi = float(np.nanmean(phi_eff[baseline_lo:baseline_hi]))
            baseline_R2_seeds.append(base_R2)
            baseline_phi_seeds.append(base_phi)

            if base_R2 > 0:
                a_star_R2_seeds.append(
                    find_a_star(A, R2, r2_threshold_factor * base_R2, direction="below")
                )
            if base_phi > 0:
                a_star_phi_seeds.append(
                    find_a_star(A, phi_eff, phi_threshold_factor * base_phi, direction="above")
                )

        a_star_R2[i, j] = float(np.nanmean(a_star_R2_seeds)) if a_star_R2_seeds else np.nan
        a_star_phi[i, j] = float(np.nanmean(a_star_phi_seeds)) if a_star_phi_seeds else np.nan
        baseline_R2_grid[i, j] = float(np.nanmean(baseline_R2_seeds))
        baseline_phi_grid[i, j] = float(np.nanmean(baseline_phi_seeds))

print(f"Sweep complete: {n_phi}x{n_mu} cells, {num_seeds_phase6} seeds each.")

In [ ]:
print(f"{'phi':>6} | {'mu':>8} | {'A*_R2':>8} | {'A*_phi':>8} | {'baseline_R2':>12} | {'baseline_phi':>12}")
print("-" * 70)
for i, phi_v in enumerate(phi_grid):
    for j, mu_v in enumerate(mu_grid):
        print(
            f"{phi_v:>6.2f} | {mu_v:>8.3f} | "
            f"{a_star_R2[i, j]:>8.3f} | {a_star_phi[i, j]:>8.3f} | "
            f"{baseline_R2_grid[i, j]:>12.4f} | {baseline_phi_grid[i, j]:>12.4f}"
        )

## Heatmap 1: A*_R2 across (mu, phi)

Smallest adoption share at which the rolling residual OOS R^2 has fallen to 50% of its low-adoption baseline. Darker cells mean erosion bites at lower adoption (the mechanism is stronger). Lighter cells mean the rule survives further into the run. White / NaN cells mean the threshold is not crossed inside the observable adoption range (A <= ~0.87 for the slow-diffusion regime in T = 8000).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    a_star_R2, origin="lower", cmap="viridis_r", vmin=0.0, vmax=1.0, aspect="auto"
)
ax.set_xticks(range(n_mu))
ax.set_xticklabels([f"{m:g}" for m in mu_grid])
ax.set_yticks(range(n_phi))
ax.set_yticklabels([f"{p:.2f}" for p in phi_grid])
ax.set_xlabel("mu (price-impact coefficient)")
ax.set_ylabel("phi (input AR coefficient)")
ax.set_title(
    f"A* at residual R^2 = 0.5 x baseline (averaged over {num_seeds_phase6} seeds)"
)
for i in range(n_phi):
    for j in range(n_mu):
        v = a_star_R2[i, j]
        text = "-" if not np.isfinite(v) else f"{v:.2f}"
        ax.text(j, i, text, ha="center", va="center",
                color="white" if np.isfinite(v) and v < 0.5 else "black")
fig.colorbar(im, ax=ax, label="A*")
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_06_a_star_R2_heatmap.png", dpi=150)
plt.show()

## Heatmap 2: A*_phi across (mu, phi)

Smallest adoption share at which the rolling effective phi has grown to 150% of its low-adoption baseline. This is the same mechanism viewed through the autoregressive amplification rather than the forecast erosion. Darker cells mean amplification bites earlier.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    a_star_phi, origin="lower", cmap="viridis_r", vmin=0.0, vmax=1.0, aspect="auto"
)
ax.set_xticks(range(n_mu))
ax.set_xticklabels([f"{m:g}" for m in mu_grid])
ax.set_yticks(range(n_phi))
ax.set_yticklabels([f"{p:.2f}" for p in phi_grid])
ax.set_xlabel("mu (price-impact coefficient)")
ax.set_ylabel("phi (input AR coefficient)")
ax.set_title(
    f"A* at effective phi = 1.5 x baseline (averaged over {num_seeds_phase6} seeds)"
)
for i in range(n_phi):
    for j in range(n_mu):
        v = a_star_phi[i, j]
        text = "-" if not np.isfinite(v) else f"{v:.2f}"
        ax.text(j, i, text, ha="center", va="center",
                color="white" if np.isfinite(v) and v < 0.5 else "black")
fig.colorbar(im, ax=ax, label="A*")
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_06_a_star_phi_heatmap.png", dpi=150)
plt.show()

## Save numeric summary

Stored as a small npz with the two A* matrices, the parameter grids, the baselines, and the run inputs.

In [ ]:
np.savez(
    f"{data_dir}/phase_06_a_star_grid.npz",
    mu_grid=mu_arr,
    phi_grid=phi_arr,
    a_star_R2=a_star_R2,
    a_star_phi=a_star_phi,
    baseline_R2_grid=baseline_R2_grid,
    baseline_phi_grid=baseline_phi_grid,
    forecast_window=np.array(forecast_window),
    eval_window=np.array(eval_window),
    risk_scale=np.array(risk_scale),
    q_cap=np.array(q_cap),
    adoption_pi=np.array(adoption_pi),
    r2_threshold_factor=np.array(r2_threshold_factor),
    phi_threshold_factor=np.array(phi_threshold_factor),
    T=np.array(T),
    N=np.array(N),
    num_seeds=np.array(num_seeds_phase6),
    base_seed=np.array(base_seed),
)

## Done when

- Both heatmaps are populated for the bulk of the (mu, phi) grid (a few NaN cells at extreme corners are acceptable: e.g. very small phi means baseline R^2 is essentially noise so the relative threshold is meaningless).
- The heatmaps tell a coherent monotonic story: larger mu and larger phi push A* down (erosion bites earlier), smaller mu and smaller phi push it up (or out of range entirely).
- The two heatmaps are visually consistent (same cells dark in both): the residual-R^2 erosion and the effective-phi amplification are two views of the same mechanism, so they should track.

Phase 7 will add transaction costs and the absolute *net profit crosses zero* version of A* becomes operational; rerun the same sweep with the cost term and a third heatmap appears that the cost-free baseline can't produce.